# Initialisation

Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import phobos
import time
import src.N4x4_T8_characterisation as characterisation

✅ BMC lib found. Running in control mode.


Delcarations

In [ ]:
arch = phobos.Arch6()
path = phobos.archive.new("N4x4-T8_characterisation", verbose=True)

Archive created at /mnt/disk1/archive/2026-03-09/1_N4x4-T8_characterisation


Initial setup

In [ ]:
phobos.XPOW().turn_off()
phobos.FilterWheel().move(4)

Filter Wheel connected on port /dev/ttyUSBthorlabs
FILT - Move to position 4


# Calibration

Injection

In [ ]:
metadata = phobos.DM().calibrate_injection(plot=True, verbose=True)
metadata['figure'].savefig(path / "injection_calibration.png")
phobos.DM().balanced()

TOPAs

In [ ]:
data, metadata = arch.phase_calibration(plot=True, return_metadata=True)
metadata['figure1'].savefig(path / "phase_calibration.png")
metadata['figure2'].savefig(path / "phase_calibration_verification.png")
phobos.XPOW().turn_off()

In [ ]:
time.sleep(5*60) # Wait for the system to cool down

# Model solving

Model
$$
O = |C_{out} \cdot M \cdot C_{in} \cdot E|^2

In [ ]:
# MMI transfer function
M = 1/np.sqrt(4) * np.matrix([
    [1,  1,   1, 1],
    [1 -1j,  1j, 1],
    [1, 1j, -1j, 1],
    [1, -1,  -1, 1]
])

def electric_field_model_ideal(E):
    return M @ E

def intensity_model_ideal(E):
    return np.abs(electric_field_model_ideal(E))**2

Systematic scan

In [ ]:
data, metadata = characterisation.systematic_scan(arch, plot=True, return_metadata=True)
for i in range(5):
    metadata[f'figure{i}'].savefig(path / f"systematic_scan_{i}_input.png")

Solve matrix model

In [ ]:
# outs = arch.solve_matrix(data_path='./generated/architecture_characterization/4-Port_MMI_Active/20251208_164250/characterization_data.npz')
Cout, Cin, Eon, Eoff, metadata = characterisation.fit_model(arch, plot=True, return_metadata=True)
for i in range(5):
    metadata[f'figure{i}'].savefig(path / f"model_fitting_{i}_input.png")
np.savez(path / "model_parameters.npz", Cout=Cout, Cin=Cin, Eon=Eon, Eoff=Eoff)

def electric_field_model_real(E):
    return Cout @ M @ Cin @ E

def intensity_model_real(E):
    return np.abs(electric_field_model(E))**2

Display phasors

In [ ]:
# Ideal
fig, axs = phise.laugiergram(model=electric_field_model_ideal, inputs=Ein, title="Ideal phasors")
fig.savefig(path / "ideal_phasors.png")

# Raw
fig, axs = phise.laugiergram(model=electric_field_model_fitted, inputs=Ein, title="Fitted phasors")
fig.savefig(path / "fitted_phasors.png")

# Calibrated
fig, axs = phise.laugiergram(model=electric_field_model_calibrated, inputs=Ein, title="Calibrated phasors")
fig.savefig(path / "calbirated_phasors.png")

Display fitted parameters

In [ ]:
utils.latexio.display_complex_vector(Eon, name="Eon")
utils.latexio.display_complex_vector(Eoff, name="Eoff")
utils.latexio.display_complex_matrix(Cin, name="Cin")
utils.latexio.display_complex_matrix(Cout, name="Cout")

---

# Null Calibration

In [ ]:
phobos.DM().balanced()
phobos.XPOW().turn_off()

In [ ]:
meas = []
for _ in range(100):
    meas.append(np.sum(phobos.Cred3().get_outputs(flux_mode='sum')))
total_flux = np.mean(meas)
total_flux

In [ ]:
bright_output=0
phobos.FilterWheel().move(2)

img_before_calib = phobos.Cred3().get_image()
img_before_calib[img_before_calib < 0] = 1e-10  # Avoid log of zero
plt.imshow(np.log(img_before_calib), cmap='jet', vmin=0, vmax=np.percentile(img_before_calib, 99))
plt.title('Image Before Calibration')

calibration_res = phobos.Arch6().null_calibration_gen(bright_output=bright_output, use_dm=False, plot=True, total_flux=total_flux, figsize=(7, 15))

img_after_calib = phobos.Cred3().get_image()
img_after_calib[img_after_calib < 0] = 1e-10  # Avoid log of zero
plt.imshow(np.log(img_after_calib), cmap='jet', vmin=0, vmax=np.percentile(img_after_calib, 99))
plt.title('Image After Calibration')

In [ ]:
phases = calibration_res["phases"]
history = {"depth": calibration_res["depth_evol"], "shifters": calibration_res["phases_evol"]}
fig_null_calibration = calibration_res.get("figure", None)

In [ ]:
phobos.Arch6().set_phases(phases)
phobos.Cred3().get_outputs(flux_mode='sum')

In [ ]:
calibrated_phasors = phobos.Arch6().predict_phasors(injected_phases=phases)

In [ ]:

def get_distrib(N=1000, rms=0):
    phobos.Arch6().set_phases(phases)
    phobos.DM().balanced()

    outs = np.zeros((N, 4), dtype=complex)
    kernel = np.zeros((N,), dtype=complex)
    for i in range(N):
        for seg, ptt in phobos.Config().get('dm.ptt_balanced').items():
            ptt[0] += np.random.normal(0, rms)  # Add noise to DM commands
            phobos.Segment(int(seg)).set_ptt(*ptt)
        outs[i] = phobos.Cred3().get_outputs(flux_mode='sum')
        kernel[i] = outs[i,(bright_output+1)%4] - outs[i,(bright_output+2)%4]

    # Histograms
    import matplotlib.pyplot as plt
    distribution_fig, axs = plt.subplots(1, 5, figsize=(20, 5), tight_layout=True)
    axs[0].hist(outs[:,bright_output], bins=30, color='blue', alpha=0.7)
    axs[0].set_title('Bright')
    axs[1].hist(outs[:,(bright_output+1)%4], bins=30, color='orange', alpha=0.7)
    axs[1].set_title('Dark 1')
    axs[2].hist(outs[:,(bright_output+2)%4], bins=30, color='green', alpha=0.7)
    axs[2].set_title('Dark 2')
    axs[3].hist(outs[:,(bright_output+3)%4], bins=30, color='red', alpha=0.7)
    axs[3].set_title('Null')
    axs[4].hist(kernel, bins=30, color='purple', alpha=0.7)
    axs[4].set_title('Kernel')
    for ax in axs:
        ax.set_xlabel('Integrated flux (ADU)')
        ax.set_ylabel('Occurrences')
    plt.show()

    print("Median of null depth:")
    bright = np.median(np.abs(outs[:,2]))
    null = np.median(np.abs(outs[:,1]))
    dark_1 = np.median(np.abs(outs[:,0]))
    dark_2 = np.median(np.abs(outs[:,3]))
    kernel_med = np.median(np.abs(kernel))
    print(f"   Null: {null/bright:.2e}")
    print(f"   Dark 1: {dark_1/bright:.2e}")
    print(f"   Dark 2: {dark_2/bright:.2e}")
    print(f"   Kernel: {kernel_med/bright:.2e}")

    # Combine kernels and outs
    distribs = np.hstack((outs, kernel.reshape(-1, 1)))
    return distribs

In [ ]:
rms_values = [0, 1, 2, 5, 10, 20, 50, 100, 200]

distribs = []
for rms in rms_values:
    print(f"\nRMS Noise: {rms} nm")
    distrib = get_distrib(N=1000, rms=rms)
    distribs.append(distrib)


In [ ]:
# Violin plots of distributions vs piston RMS for Bright, Null, Dark1, Dark2 and Kernel
# Uses `distribs` and `rms_values` produced by previous cells, and `bright_output` defined earlier.

# Map outputs
bright_idx = bright_output
dark1_idx = (bright_idx + 1) % 4
dark2_idx = (bright_idx + 2) % 4
null_idx  = (bright_idx + 3) % 4
kernel_idx = 4  # last column in distribs is the kernel

output_map = [
    ("Bright", bright_idx),
    ("Dark 1", dark1_idx),
    ("Dark 2", dark2_idx),
    ("Null", null_idx),
    ("Kernel", kernel_idx),
]

# Prepare data: list of arrays per RMS for each output (use absolute value)
data_per_output = {}
eps = 1e-12  # to avoid log(0) issues if using log scale
for name, idx in output_map:
    series = [np.abs(d[:, idx]) + eps for d in distribs]
    data_per_output[name] = series

# Plot violins
fig, axs = plt.subplots(len(output_map), 1, figsize=(10, 30), sharey=True)
for ax, (name, _) in zip(axs, output_map):
    series = data_per_output[name]
    parts = ax.violinplot(series, positions=rms_values, showmeans=False, showmedians=False, widths=0.8)
    # Style violins
    for pc in parts['bodies']:
        pc.set_facecolor('#9fbff7')
        pc.set_edgecolor('black')
        pc.set_alpha(0.8)
    # Overlay median
    medians = [np.median(s) for s in series]
    ax.plot(rms_values, medians, marker='o', color='k', linestyle='-', linewidth=1, label='median')
    ax.set_title(name)
    ax.set_xlabel('Piston RMS (nm)')
    ax.grid(axis='y', alpha=0.3)
    ax.set_xscale('log')

axs[0].set_ylabel('Integrated flux (ADU)')
# Optionally use log scale for better dynamic range visualization
axs[0].set_yscale('log')

plt.suptitle('Distributions vs Piston RMS (violin plots)')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
phobos.XPOW().turn_off()
phobos.DM().max()

---
# Save results

In [ ]:
# Turn off
import phobos
phobos.XPOW().turn_off()
phobos.DM().flat()  # Reset injection segments to (piston, tip, tilt) = (0, 0, 0)

model_path = phobos.archive.new("Model Caracterization")
model_path

In [ ]:
# --- MMI characterization ---

# data
np.save(model_path / "A_model", phobos.Arch6().A_model)
np.save(model_path / "Cin_model", phobos.Arch6().C_model)
np.save(model_path / "Eoff_model", phobos.Arch6().Eoff_model)
np.save(model_path / "Eon_model", phobos.Arch6().Eon_model)

# figures
for i, fig in enumerate(solved_matrix.get('figures', [])):
    fig.savefig(model_path / f"Model Fitting with {i+1} inputs.png", dpi=200, bbox_inches="tight")

solved_phasors['figure'].savefig(model_path / "raw_phasors.png", dpi=200, bbox_inches="tight")
calibrated_phasors['figure'].savefig(model_path / "calibrated_phasors.png", dpi=200, bbox_inches="tight")

In [ ]:
# --- Null Calibration ---

null_path = phobos.archive.new("Null Calibration")
print(null_path)

dark1_output = (bright_output + 1) % 4
dark2_output = (bright_output + 2) % 4
null_output = (bright_output + 3) % 4

# data
np.save(null_path / "Distribs.npy", distribs)

readme = """
Distrib.npy contient les distributions pour chaque observable et pour chaque régime de piston rms.

distribs[piston_rms, observable, sample]

les piston_rms sont = [0, 1, 2, 5, 10, 50, 100, 200]
les observables sont [Bright, Dark1, Dark2, Null, Kernel]
"""
# Save readme
with open(null_path / "README.txt", "w") as f:
    f.write(readme)

# figures
fig_null_calibration.savefig(null_path / "null_calibration.png", dpi=200, bbox_inches="tight")